Run this cell first. It downloads the helper module `rl_lab.py` and the prepared rollouts from the course repository, unless they are already next to the notebook.

In [ ]:
import os, urllib.request
ASSETS = "https://raw.githubusercontent.com/BRAIn-Lab-teaching/ADDITIONAL-CHAPTERS-NLP-COURSE/main/week07/seminar/"
for f in ['rl_lab.py', 'rollouts/samples_base.jsonl.gz', 'rollouts/samples_rl.jsonl.gz', 'rollouts/samples_sft.jsonl.gz', 'rollouts/traces_r1trace.jsonl.gz']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(ASSETS + f, f)
print("rl_lab.py and", len(os.listdir("rollouts")), "rollout files are ready")


## Setup

Run this cell first — it is used later to render the interactive plots inline.

In [ ]:
def _show_html(path, height=650):
    """Embed an already-written standalone HTML file inline as an output.
    Uses srcdoc (content inlined into the iframe) rather than src (a file
    path/URL) so it renders regardless of how the notebook viewer resolves
    external paths."""
    import html as _html
    import warnings
    from IPython.display import HTML, display
    warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
    with open(path, encoding="utf-8") as f:
        content = f.read()
    display(HTML(f'<iframe srcdoc="{_html.escape(content)}" '
                 f'width="100%" height="{height}" style="border:none"></iframe>'))


# Seminar 7. RL and reasoning, by hand

Today we take a few claims from the lecture and check them on models small
enough to run right here: Qwen2.5-0.5B-Instruct and Qwen3-0.6B.

1. **One GRPO step.** Sample eight answers to each of two questions, turn
   their rewards into advantages and a loss, take one optimizer step and see
   which answers became more likely.
2. **pass@1 vs pass@k.** Estimate pass@k, draw the answers as a tree, then
   compare a base model, its RL version and a distilled model.
3. **Inside one trace.** Follow a reasoning trace token by token: where the
   model hesitates, when the right answer first shows up, what it writes
   after that.
4. **Predict the reward.** Three reward functions: you guess what they
   reward, then the cell computes it.

Three cells have gaps that we fill in together (the group advantage, the
GRPO loss, the pass@k estimate); until then they raise `NotImplementedError`.

Everything you see is computed in this notebook. Parts 2 and 3 end with the
same analysis on a larger set of answers from 1.5B models, generated in
advance by `gpu/make_rollouts.py`; if `rollouts/` is empty, those cells skip.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from rl_lab import DEVICE, POLICY, THINKER, load_model

OUT = Path("out")
OUT.mkdir(exist_ok=True)
ROLLOUTS = Path("rollouts")

print(f"device: {DEVICE}, torch threads: {torch.get_num_threads()}")
prepared = sorted(p.name for p in ROLLOUTS.glob("*.jsonl.gz")) if ROLLOUTS.exists() else []
print("prepared rollouts:", ", ".join(prepared) if prepared else "none (large-scale cells will skip)")

The first run downloads both models from the Hugging Face Hub (about 1 GB and
1.5 GB); after that they stay in memory until the end of the notebook.

In [ ]:
t0 = time.time()
for name in (POLICY, THINKER):
    tok, model = load_model(name)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"{name}: {n_params:.0f}M parameters")
print(f"loaded in {time.time() - t0:.0f} s")

---

## 1. One GRPO step by hand

A GRPO step is made of four pieces: a group of answers to one question, a
reward for each, an advantage computed inside the group, and a clipped loss
over tokens. We build them one by one on a real model and finish with one
optimizer step.

The questions are GSM8K word problems with an integer answer. The reward is 1
if the number in the last `\boxed{}` matches the reference, and 0 otherwise.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from rl_lab import DEVICE, POLICY, completion_logprobs, extract_answer, load_model, sample_group

OUT = Path("out")
OUT.mkdir(exist_ok=True)

# two GSM8K test problems (#103, #109) on which the 0.5B model is right part of the time
PROBLEMS = [
    {"question": "A fog bank rolls in from the ocean to cover a city. It takes 10 minutes to cover every 3 miles of the city. If the city is 42 miles across from the oceanfront to the opposite inland edge, how many minutes will it take for the fog bank to cover the whole city?",
     "gold": 140},
    {"question": "A local gas station is selling gas for $3.00 a gallon.  An app company is offering $.20 cashback per gallon if you fill up at this station.  If someone buys 10 gallons of gas, how much with their gas be, after the cashback rewards?",
     "gold": 28},
]
G = 8   # answers per question

### 1.1 A group of answers

Eight samples per question at temperature 1: a few seconds on a GPU, one to
three minutes per question on a CPU.

In [ ]:
t0 = time.time()
groups = [sample_group(POLICY, p["question"], p["gold"], G=G, seed=i)
          for i, p in enumerate(PROBLEMS)]
print(f"sampled {G * len(PROBLEMS)} answers in {time.time() - t0:.0f} s\n")
for i, (p, grp) in enumerate(zip(PROBLEMS, groups)):
    print(f"question {i}: gold = {p['gold']}")
    for j, s in enumerate(grp):
        print(f"  answer {j}: extracted {extract_answer(s.text)!s:>8}  "
              f"{'correct' if s.correct else 'wrong  '}  {s.n_tokens:4d} tokens")

Here is one of them in full, a correct one if the groups have any:

In [ ]:
shown = next((s for grp in groups for s in grp if s.correct), groups[0][0])
print(shown.text)

### 1.2 From rewards to advantages

GRPO has no critic. An answer is compared with the mean reward of its own
group, and the difference is divided by the group's standard deviation so that
easy and hard questions end up on one scale. Dr. GRPO keeps the subtraction
and drops the division.

In [ ]:
def group_advantage(rewards: torch.Tensor, scale_by_std: bool = True, eps: float = 1e-6):
    # TODO(live-coding): subtract the group mean; divide by the group std if scale_by_std (~4 lines)
    raise NotImplementedError('subtract the group mean; divide by the group std if scale_by_std')


rewards = [torch.tensor([float(s.correct) for s in grp]) for grp in groups]
for i, r in enumerate(rewards):
    print(f"question {i}: rewards   {r.tolist()}")
    print(f"            GRPO adv  {[round(a, 2) for a in group_advantage(r).tolist()]}")
    print(f"            Dr. GRPO  {[round(a, 2) for a in group_advantage(r, False).tolist()]}")

When all answers in a group get the same reward, every advantage is zero: the
question costs G samples and adds nothing to the gradient.

How likely is that before we sample? Let p be the chance that a single answer
of the model to this question is correct. All G answers agree with
probability p^G + (1 − p)^G: questions the model almost always or almost never
solves waste most of their groups.

In [ ]:
p = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for g in (4, 8, 16):
    ax.plot(p, p ** g + (1 - p) ** g, label=f"G = {g}")
ax.set_xlabel("p, chance that one answer is correct")
ax.set_ylabel("P(all G rewards equal)")
ax.set_title("Before sampling: chance that a group gives no signal")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "01_dead_groups.png", dpi=130)
plt.show()
for i, r in enumerate(rewards):
    k = int(r.sum())
    verdict = "no signal: all advantages are zero" if k in (0, G) else "has a signal"
    print(f"question {i}: {k} of {G} correct, the group {verdict}")

### 1.3 The loss

For answer i and its token t, with ratio ρ = π_θ / π_old:

$$\ell_{i,t} = -\Big[\min\big(\rho_{i,t}\hat A_i,\ \mathrm{clip}(\rho_{i,t}, 1-\varepsilon, 1+\varepsilon)\hat A_i\big) - \beta\,\mathrm{KL}_{i,t}\Big],\qquad \mathrm{KL}_{i,t} = e^{\,r} - r - 1,\ \ r = \log\pi_{\mathrm{ref}} - \log\pi_\theta$$

GRPO first averages over the tokens of each answer (a factor 1/|o_i|) and
then over answers; Dr. GRPO divides the sum over all tokens by one constant.

In [ ]:
def grpo_loss(logp, old_logp, ref_logp, adv, mask, eps=0.2, beta=0.04, norm="grpo"):
    # TODO(live-coding): clipped surrogate per token, k3 KL to the reference, then normalize (~9 lines)
    raise NotImplementedError('clipped surrogate per token, k3 KL to the reference, then normalize')

How hard does each token pull on the gradient? At the first step ρ = 1 and
KL = 0, so a token of answer i has weight A_i / |o_i| under GRPO and
A_i / (G · T_max) under Dr. GRPO. Look at the wrong answers of different
lengths.

In [ ]:
qi = max(range(len(groups)), key=lambda i: rewards[i].std().item())   # the most mixed group
grp, adv = groups[qi], group_advantage(rewards[qi])
lens = torch.tensor([s.n_tokens for s in grp], dtype=torch.float)
w_grpo = adv / lens
w_dr = adv / (G * lens.max())
order = torch.argsort(lens)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True)
for ax, w, title in ((axs[0], w_grpo, "GRPO: A / |o|"), (axs[1], w_dr, "Dr. GRPO: A / (G·T_max)")):
    colors = ["#2a9d8f" if grp[j].correct else "#c0392b" for j in order]
    ax.bar(range(G), w[order].numpy(), color=colors)
    ax.set_xticks(range(G), [f"{int(lens[j])}" for j in order], fontsize=8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(title)
    ax.set_xlabel("answer length, tokens (green: correct, red: wrong)")
axs[0].set_ylabel("weight per token")
fig.tight_layout()
fig.savefig(OUT / "01_token_weights.png", dpi=130)
plt.show()

wrong = [j for j in range(G) if not grp[j].correct]
if len(wrong) >= 2:
    j_s, j_l = min(wrong, key=lambda j: lens[j]), max(wrong, key=lambda j: lens[j])
    print(f"wrong answers of {int(lens[j_s])} and {int(lens[j_l])} tokens: per-token penalty "
          f"{abs(w_grpo[j_s]):.4f} vs {abs(w_grpo[j_l]):.4f} under GRPO "
          f"(×{abs(w_grpo[j_s] / w_grpo[j_l]):.1f}), equal under Dr. GRPO")

### 1.4 One optimizer step

Now the step itself, on the group with the most mixed rewards. Only the last
four transformer blocks are trained (Adam, lr 1e-6), which keeps memory low.
After the step we measure how the log-probability of every answer changed.

The answers go through the model one at a time. The GRPO loss is a mean over
answers, so the gradients of the per-answer losses, each divided by G, add up
to exactly the gradient of the whole group, and only one answer has to be in
memory at once.

Expect the printed loss to be about zero: with ρ = 1, KL = 0 and advantages
that sum to zero within the group, the loss value cancels out, but its
gradient does not.

In [ ]:
tok, model = load_model(POLICY)
blocks = model.model.layers[-4:]
saved = {k: v.detach().clone() for k, v in blocks.state_dict().items()}   # to undo the step
for p_ in model.parameters():
    p_.requires_grad_(False)
for p_ in blocks.parameters():
    p_.requires_grad_(True)
opt = torch.optim.Adam(blocks.parameters(), lr=1e-6)

with torch.no_grad():
    old_logp, mask = completion_logprobs(model, grp, tok.pad_token_id)
ref_logp = old_logp.clone()                  # the reference is the policy before the step

t0 = time.time()
model.train()
loss = 0.0
for i, s in enumerate(grp):                 # one answer at a time, to fit in GPU memory
    n = s.n_tokens
    logp_i, _ = completion_logprobs(model, [s], tok.pad_token_id)
    loss_i = grpo_loss(logp_i, old_logp[i:i + 1, :n], ref_logp[i:i + 1, :n],
                       adv[i:i + 1].to(DEVICE), mask[i:i + 1, :n]) / G
    loss_i.backward()                       # gradients add up across answers
    loss += loss_i.item()
opt.step()
opt.zero_grad()
model.eval()
with torch.no_grad():
    new_logp, _ = completion_logprobs(model, grp, tok.pad_token_id)
print(f"loss {loss:+.4f}, step took {time.time() - t0:.0f} s")

delta = ((new_logp - old_logp) * mask).sum(1)          # change of log p(answer)
corr = np.corrcoef(adv.numpy(), delta.cpu().numpy())[0, 1]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.scatter(adv.numpy(), delta.cpu().numpy(),
           c=["#2a9d8f" if s.correct else "#c0392b" for s in grp], s=40)
ax.axhline(0, color="k", lw=0.6)
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("advantage of the answer")
ax.set_ylabel("Δ log p(answer) after one step")
ax.set_title(f"One step: correlation with the advantage {corr:+.2f}")
fig.tight_layout()
fig.savefig(OUT / "01_step_effect.png", dpi=130)
plt.show()
for s, a, d in sorted(zip(grp, adv.tolist(), delta.tolist()), key=lambda x: -x[1]):
    print(f"{'correct' if s.correct else 'wrong  '}  advantage {a:+.2f}  Δ log p {d:+.2f}")

In [ ]:
# undo the step: later parts use the original model
blocks.load_state_dict(saved)
for p_ in model.parameters():
    p_.requires_grad_(False)

---

## 2. pass@1 vs pass@k

pass@k is the probability that at least one of k samples is correct. RL
training clearly raises pass@1. Does it raise pass@k for large k too? In
Yue et al. it did not: with enough samples (they went up to k = 256) the base
model caught up with the RL model and overtook it. We will see how the gap
behaves on our data.

From n samples with c correct, the unbiased estimate is

$$\text{pass@}k = 1 - \binom{n-c}{k} \Big/ \binom{n}{k}$$

(the chance that a random k-subset of the n samples contains no correct one,
subtracted from 1).

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from rl_lab import POLICY, is_correct, load_rollouts, sample_group, sentence_tree

OUT = Path("out")
OUT.mkdir(exist_ok=True)
ROLLOUTS = Path("rollouts")


def pass_at_k(n: int, c: int, k: int) -> float:
    # TODO(live-coding): 1 - C(n-c, k) / C(n, k), as a running product to stay stable (~3 lines)
    raise NotImplementedError('1 - C(n-c, k) / C(n, k), as a running product to stay stable')


# check against brute force: enumerate every k-subset of a small sample set
rng = np.random.default_rng(0)
for n, c in ((8, 3), (10, 1), (12, 7)):
    flags = np.array([1] * c + [0] * (n - c))
    for k in (1, 2, 4):
        brute = np.mean([flags[list(s)].any() for s in combinations(range(n), k)])
        assert abs(brute - pass_at_k(n, c, k)) < 1e-12
print("formula matches brute force on every k-subset")

### 2.1 Small: one question, eight answers

We start with the eight answers to the second question of part 1 (in the
notebook they are reused, not generated again).

In [ ]:
QUESTION = "A local gas station is selling gas for $3.00 a gallon.  An app company is offering $.20 cashback per gallon if you fill up at this station.  If someone buys 10 gallons of gas, how much with their gas be, after the cashback rewards?"   # GSM8K #109, the second question of part 1
GOLD = 28
small = sample_group(POLICY, QUESTION, GOLD, G=8, seed=1)      # same call as in part 1
c = sum(s.correct for s in small)
print(f"{c} of 8 correct: " + ", ".join(f"pass@{k} = {pass_at_k(8, c, k):.2f}" for k in (1, 2, 4, 8)))

**Visualization: the tree of answers.** Each level is the next sentence of
an answer. Answers whose sentences start with the same four words (list
numbering and markdown ignored) share a branch, labelled by those words. The
width of a branch is the number of answers going through it, and its colour
shows how they end: green correct, red wrong, grey both. Click a branch to
zoom in.

In [ ]:
import plotly.graph_objects as go

COLORS = {"correct": "#2a9d8f", "wrong": "#c0392b", "mixed": "#b9c2cc", "all": "#e9edf1"}


def tree_figure(texts, flags, title, height=520, n_words=4):
    t = sentence_tree(texts, ["correct" if f else "wrong" for f in flags], n_words=n_words)
    fig = go.Figure(go.Icicle(ids=t["ids"], parents=t["parents"], labels=t["names"],
                              values=t["values"], branchvalues="total",
                              marker=dict(colors=[COLORS[c] for c in t["labels"]]),
                              tiling=dict(orientation="v"), maxdepth=4,
                              hovertemplate="%{label}<br>%{value} answers<extra></extra>"))
    fig.update_layout(title=title, height=height, margin=dict(t=40, l=5, r=5, b=5))
    return fig


fig = tree_figure([s.text for s in small], [s.correct for s in small],
                  f"8 answers of {POLICY.split('/')[-1]}: {c} correct")
fig.write_html(OUT / "02_tree_small.html", include_plotlyjs="cdn")
_show_html(OUT / "02_tree_small.html", height=540)

### 2.2 At scale: base, RL and SFT models of one family

The same questions for three models of one family, on MATH-500 problems
with an integer answer: 32 answers per problem for base and RL, 16 for SFT
(its answers are several times longer).

* **base**: Qwen2.5-Math-1.5B;
* **RL**: the same base after R1-Zero-style RL (Oat-Zero, Dr. GRPO paper);
* **SFT**: DeepSeek-R1-Distill-Qwen-1.5B, distilled from R1 traces.

The files hold only the texts; correctness is computed here with the same
checker as above.

In [ ]:
MODELS = {"base": "samples_base.jsonl.gz", "RL": "samples_rl.jsonl.gz",
          "SFT": "samples_sft.jsonl.gz"}
big = {name: load_rollouts(ROLLOUTS / f) for name, f in MODELS.items()}
HAVE_BIG = all(v is not None for v in big.values())
if not HAVE_BIG:
    missing = [f for name, f in MODELS.items() if big[name] is None]
    print("no prepared rollouts (" + ", ".join(missing) + ") — skipping 2.2; "
          "generate them with gpu/make_rollouts.py")
else:
    flags = {name: [[is_correct(s["text"], r["gold"]) for s in r["samples"]] for r in rows]
             for name, rows in big.items()}
    for name, f in flags.items():
        cut = np.mean([s.get("finish") == "length" for r in big[name] for s in r["samples"]])
        print(f"{name:>4}: {len(f)} problems × {len(f[0])} answers, "
              f"mean accuracy {np.mean([np.mean(x) for x in f]):.3f}, "
              f"{cut:.0%} of answers cut by the length limit")

In [ ]:
if HAVE_BIG:
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    curves = {}
    for name, f in flags.items():
        n = len(f[0])                       # answers per problem for this model
        ks = 2 ** np.arange(int(np.log2(n)) + 1)
        curves[name] = (ks, [np.mean([pass_at_k(n, int(sum(x)), k) for x in f]) for k in ks])
        ax.plot(*curves[name], marker="o", ms=3, label=f"{name} (n = {n})")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("k")
    ax.set_ylabel("pass@k")
    ax.set_title("pass@k of three models of one family")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "02_pass_at_k.png", dpi=130)
    plt.show()
    for name, (ks, ys) in curves.items():
        print(f"{name:>4}: " + "  ".join(f"pass@{k} {y:.3f}" for k, y in zip(ks, ys)))

**How many different wrong answers?** For each problem we count the distinct
final answers among the wrong ones, using the first 16 answers of every model
so that all three are compared on the same number of samples. A model that
keeps failing the same way gives one or two; a model that wanders gives many.
Correct answers all end in the same number, so their variety is better seen
in the trees below.

In [ ]:
from rl_lab import extract_answer

if HAVE_BIG:
    m = min(len(f[0]) for f in flags.values())      # same number of answers for every model
    wrong = {}
    for name, rows in big.items():
        per_problem = [len({extract_answer(s["text"]) for s, ok in zip(r["samples"][:m], f[:m])
                            if not ok})
                       for r, f in zip(rows, flags[name])]
        wrong[name] = np.mean(per_problem)
        print(f"{name:>4}: {wrong[name]:.1f} distinct wrong final answers per problem (of {m} answers)")
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.bar(list(wrong), list(wrong.values()), color=["#1f77b4", "#ff7f0e", "#2ca02c"])
    ax.set_ylabel("distinct wrong answers per problem")
    ax.set_title(f"Out of the first {m} answers per problem")
    fig.tight_layout()
    fig.savefig(OUT / "02_wrong_answers.png", dpi=130)
    plt.show()

The tree from 2.1 again, for one problem and all three models. The problem is
picked automatically: the one where the base model is right closest to 30%
of the time. Here sentences must match exactly: with the four-word rule the
template openers of the SFT model ("Okay, so I need to ...") would merge
most of its answers into one branch.

In [ ]:
if HAVE_BIG:
    from plotly.subplots import make_subplots
    accs = np.array([np.mean(f) for f in flags["base"]])
    pi = int(np.argmin(np.abs(accs - 0.3)))
    print(f"problem {big['base'][pi]['pid']}: base accuracy {accs[pi]:.2f}")
    print(big["base"][pi]["question"][:400])
    fig = make_subplots(rows=1, cols=3, specs=[[{"type": "domain"}] * 3],
                        subplot_titles=list(big))
    for col, name in enumerate(big, 1):
        r = big[name][pi]
        tr = tree_figure([s["text"] for s in r["samples"]][:m], flags[name][pi][:m], "",
                         n_words=None).data[0]
        tr.maxdepth = 3
        fig.add_trace(tr, row=1, col=col)
    fig.update_layout(height=520, margin=dict(t=40, l=5, r=5, b=5))
    fig.write_html(OUT / "02_tree_big.html", include_plotlyjs="cdn")
    _show_html(OUT / "02_tree_big.html", height=540)

---

## 3. Inside one reasoning trace

We generate one answer of Qwen3-0.6B in thinking mode and record, at every
token, the entropy of the distribution it was sampled from. Two questions
from the lecture:

* **Where is the model uncertain?** In the Qwen3-8B study about half of the
  tokens had entropy below 0.01, and the uncertain ones were mostly "forks"
  such as wait, but, so, maybe.
* **When does the answer appear?** Where the right value is first written,
  and how much the model writes after that.

In [ ]:
import html
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from rl_lab import THINKER, extract_answer, generate_trace, is_correct, load_rollouts

OUT = Path("out")
OUT.mkdir(exist_ok=True)
ROLLOUTS = Path("rollouts")

QUESTION = "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"   # GSM8K #0
GOLD = 18

t0 = time.time()
trace = generate_trace(THINKER, QUESTION, max_new_tokens=1500, seed=0)
tokens, H = trace["tokens"], np.array(trace["entropy"])
print(f"{len(tokens)} tokens in {time.time() - t0:.0f} s; "
      f"final answer {extract_answer(trace['text'])} (gold {GOLD})")

### 3.1 Where the model is uncertain

In [ ]:
print(f"tokens with entropy < 0.01 nats: {np.mean(H < 0.01):.0%}   (Qwen3-8B in the paper: ~50%)")
print(f"80th percentile of entropy:      {np.quantile(H, 0.8):.2f}   (Qwen3-8B in the paper: 0.672)")

by_word = defaultdict(list)
for t, h in zip(tokens, H):
    by_word[t.strip().lower()].append(h)
common = sorted(((w, np.mean(v), len(v)) for w, v in by_word.items() if w and len(v) >= 4),
                key=lambda x: -x[1])
print("\nwords with the highest mean entropy (seen at least 4 times):")
for w, h, n in common[:12]:
    print(f"  {w!r:>14}  {h:.2f}  ×{n}")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(H, bins=60, color="#5a6676")
ax.set_yscale("log")
ax.set_xlabel("entropy of the next-token distribution, nats")
ax.set_ylabel("tokens")
ax.set_title(f"Entropy of each of the {len(H)} tokens")
fig.tight_layout()
fig.savefig(OUT / "03_entropy_hist.png", dpi=130)
plt.show()

### 3.2 When the answer appears

We look for the first place where the reference value appears in the trace
as a standalone number, and measure how much of the thinking comes after it.

In [ ]:
def first_mention(tokens, gold):
    """Index of the token at which the gold value is first written."""
    pat = re.compile(rf"(?<![\d.]){re.escape(str(int(gold)))}(?![\d]|\.\d)")
    text, ends = "", []
    for t in tokens:
        text += t
        ends.append(len(text))
    m = pat.search(text)
    if not m:
        return None
    return next(i for i, e in enumerate(ends) if e >= m.end())


def think_end(tokens):
    text = ""
    for i, t in enumerate(tokens):
        text += t
        if "</think>" in text:
            return i
    return len(tokens) - 1


i_first, i_end = first_mention(tokens, GOLD), think_end(tokens)
if i_first is None:
    print("the gold value never appears in this trace")
else:
    after = (i_end - i_first) / max(i_end, 1)
    n_wait = sum(t.strip().lower() in ("wait", "but") for t in tokens[i_first:i_end])
    print(f"first correct value at token {i_first} of {i_end} thinking tokens; "
          f"{after:.0%} of the thinking comes after it, with {n_wait} 'wait'/'but' in that part")

**Visualization: replaying the trace.** Press play to watch the trace being
written. Darker tokens had higher entropy; the green frame marks the first
correct value, the grey one the end of thinking. What does the model do
between the two?

In [ ]:
def trace_html(tokens, H, i_first, i_end, title):
    keep = [i for i, t in enumerate(tokens) if not (t.startswith("<|") and t.endswith("|>"))]
    pos = {old: new for new, old in enumerate(keep)}            # special tokens dropped
    i_first, i_end = pos.get(i_first), pos.get(i_end)
    tokens, H = [tokens[i] for i in keep], np.asarray(H)[keep]
    hmax = max(float(np.quantile(H, 0.99)), 1e-6)
    smooth = np.convolve(H, np.ones(15) / 15, mode="same")     # 15-token moving average
    spans = []
    for i, (t, h) in enumerate(zip(tokens, H)):
        a = min(h / hmax, 1.0)
        style = f"background:rgba(133,76,101,{a:.2f})"
        if i == i_first:
            style += ";outline:2px solid #2a9d8f"
        if i == i_end:
            style += ";outline:2px solid #888"
        spans.append(f'<span id="t{i}" style="{style}" title="H={h:.2f}">'
                     f'{html.escape(t).replace(chr(10), "<br>")}</span>')
    pts = " ".join(f"{i / max(len(H) - 1, 1) * 1000:.1f},{60 - min(h / hmax, 1) * 55:.1f}"
                   for i, h in enumerate(smooth / max(smooth.max(), 1e-6) * hmax))
    marks = "".join(f'<line x1="{x / max(len(H) - 1, 1) * 1000:.1f}" x2="{x / max(len(H) - 1, 1) * 1000:.1f}" '
                    f'y1="0" y2="60" stroke="{c}" stroke-width="3"/>'
                    for x, c in ((i_first, "#2a9d8f"), (i_end, "#888")) if x is not None)
    return f"""<!doctype html><meta charset="utf-8">
<style>body{{font:13px/1.55 Menlo,Consolas,monospace;margin:10px;color:#17202e;background:#fff}}
#txt span{{visibility:hidden}} #txt span.on{{visibility:visible}}
button{{font:13px sans-serif;margin-right:8px}}</style>
<div style="font:600 14px sans-serif;margin-bottom:6px">{html.escape(title)}</div>
<button onclick="play()">▶ play</button><button onclick="show(N)">show all</button>
<input id="sl" type="range" min="0" max="{len(tokens)}" value="0" style="width:60%" oninput="show(+this.value)">
<svg viewBox="0 0 1000 60" preserveAspectRatio="none" style="width:100%;height:60px;margin:6px 0;background:#f4f6f8">
<polyline points="{pts}" fill="none" stroke="#854c65" stroke-width="1"/>{marks}
<rect id="cur" x="0" y="0" width="2" height="60" fill="#0673a0"/></svg>
<div style="font:12px sans-serif;color:#5a6676;margin-bottom:6px">line: entropy, 15-token average ·
<span style="color:#2a9d8f">■</span> first correct value · <span style="color:#888">■</span> end of thinking ·
shade of a token: its entropy (hover for the value)</div>
<div id="txt" style="white-space:normal;max-height:430px;overflow:auto">{''.join(spans)}</div>
<script>
const N={len(tokens)}, sp=document.querySelectorAll('#txt span'); let timer=null;
function show(k){{for(let i=0;i<N;i++)sp[i].classList.toggle('on',i<k);
 document.getElementById('sl').value=k;document.getElementById('cur').setAttribute('x',k/N*1000);
 if(k>0&&k<=N)sp[k-1].scrollIntoView({{block:'nearest'}});}}
function play(){{clearInterval(timer);let k=0;timer=setInterval(()=>{{show(++k);if(k>=N)clearInterval(timer);}},15);}}
show(0);
</script>"""


(OUT / "03_trace.html").write_text(
    trace_html(tokens, H, i_first, i_end, f"{THINKER.split('/')[-1]} · gold {GOLD}"),
    encoding="utf-8")
_show_html(OUT / "03_trace.html", height=620)

### 3.3 At scale: many traces of a 1.5B reasoning model

Traces of DeepSeek-R1-Distill-Qwen-1.5B on MATH-500 problems with an integer
answer, four per problem, with the entropy at every token. Here the entropy
is computed from the 20 most likely tokens only, so it comes out slightly
lower than the exact value.

We find the first correct value by searching the trace for the number, which
works only when the number cannot turn up by accident: the answer must have
at least two digits and must not appear in the question.

In [ ]:
def clear_gold(question, gold):
    """Only answers that cannot show up by accident: at least two digits and
    not already in the question (a "3" or a "4" appears in any computation)."""
    return abs(gold) >= 10 and first_mention([question], gold) is None


traces = load_rollouts(ROLLOUTS / "traces_r1trace.jsonl.gz")
if traces is None:
    print("no prepared traces (traces_r1trace.jsonl.gz) — skipping 3.3; "
          "generate them with gpu/make_rollouts.py")
else:
    shares, all_h, by_tok = [], [], defaultdict(list)
    n_tr = n_ok = 0
    for r in traces:
        for s in r["samples"]:
            n_tr += 1
            toks, h = s["tokens"], np.array(s["entropy"])
            all_h.append(h)
            for t, e in zip(toks, h):
                by_tok[t.strip().lower()].append(e)
            if not is_correct(s["text"], r["gold"]):
                continue
            n_ok += 1
            if not clear_gold(r["question"], r["gold"]):
                continue
            i_f, i_e = first_mention(toks, r["gold"]), think_end(toks)
            if i_f is not None and i_f < i_e:
                shares.append((i_e - i_f) / i_e)
    all_h = np.concatenate(all_h)
    print(f"{n_tr} traces, {n_ok} with a correct final answer, {len(all_h):,} tokens; "
          f"{len(shares)} traces with a two-digit answer absent from the question")
    print(f"tokens with entropy < 0.01: {np.mean(all_h < 0.01):.0%}; "
          f"80th percentile of entropy: {np.quantile(all_h, 0.8):.2f}")

    fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
    axs[0].hist(shares, bins=20, range=(0, 1), color="#2a9d8f")
    if shares:
        axs[0].axvline(np.median(shares), color="k", ls="--", lw=1)
        axs[0].set_title(f"median {np.median(shares):.0%} of the trace comes after the answer")
    axs[0].set_xlabel("share of thinking after the first correct value")
    axs[0].set_ylabel("correct traces")
    min_count = 50 if len(all_h) > 100_000 else 5
    common = sorted(((t, np.mean(v)) for t, v in by_tok.items() if len(v) >= min_count and t),
                    key=lambda x: -x[1])[:15]
    if common:
        names, vals = zip(*common)
        axs[1].barh(range(len(vals)), vals[::-1], color="#854c65")
        axs[1].set_yticks(range(len(vals)), [repr(n) for n in names[::-1]], fontsize=8)
    axs[1].set_xlabel("mean entropy at this token, nats")
    axs[1].set_title(f"Most uncertain frequent tokens (≥ {min_count} occurrences)")
    fig.tight_layout()
    fig.savefig(OUT / "03_traces_at_scale.png", dpi=130)
    plt.show()

---

## 4. Predict the reward

RL maximizes the reward function exactly as it is written, loopholes
included. Below are three reward functions, each with a question. **Answer it
before you run the cell underneath**: the cell computes the result on real
outputs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from rl_lab import POLICY, chat_prompt, load_model, sample_group

OUT = Path("out")
OUT.mkdir(exist_ok=True)

QUESTION = "A local gas station is selling gas for $3.00 a gallon.  An app company is offering $.20 cashback per gallon if you fill up at this station.  If someone buys 10 gallons of gas, how much with their gas be, after the cashback rewards?"   # GSM8K #109, the second question of part 1
GOLD = 28

### Card 1. A length penalty

To make answers shorter we add a penalty: r = correct − λ · (number of
tokens). The model can either reason step by step or give the number right
away. We measure both on the second question of part 1, eight answers each;
for the second behaviour the system prompt asks for the number only.

**Question:** above which λ does RL prefer answering right away, and what
happens to accuracy then? Estimate λ from the two printed lines first, then
run the plot.

In [ ]:
DIRECT = "Give only the final answer as a number in \\boxed{}. Do not explain."
reason = sample_group(POLICY, QUESTION, GOLD, G=8, seed=1)       # the group from part 1
tok, model = load_model(POLICY)
direct = sample_group(POLICY, QUESTION, GOLD, G=8, seed=12, max_new_tokens=32, system=DIRECT)
stats = {}
for name, grp in (("reason", reason), ("direct", direct)):
    acc = np.mean([s.correct for s in grp])
    ln = np.mean([s.n_tokens for s in grp])
    stats[name] = (acc, ln)
    print(f"{name:>6}: accuracy {acc:.2f}, mean length {ln:.0f} tokens")

In [ ]:
lam = np.linspace(0, 0.01, 201)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for name, (acc, ln) in stats.items():
    ax.plot(lam, acc - lam * ln, label=f"{name} (acc {acc:.2f}, {ln:.0f} tokens)")
(acc_r, ln_r), (acc_d, ln_d) = stats["reason"], stats["direct"]
if ln_r > ln_d and acc_r > acc_d:
    lam_star = (acc_r - acc_d) / (ln_r - ln_d)
    ax.axvline(lam_star, color="k", ls="--", lw=1)
    ax.set_title(f"expected reward; the lines cross at λ = {lam_star:.4f}")
else:
    ax.set_title("expected reward; reasoning was not more accurate in this sample")
ax.set_xlabel("λ (penalty per token)")
ax.set_ylabel("expected reward")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "04_length_penalty.png", dpi=130)
plt.show()

<details><summary>What practice does about it</summary>

Once λ exceeds the accuracy gained per token of reasoning, answering without
reasoning maximizes the reward. Recipes that train for shorter reasoning
compare lengths within a group of answers to the same question and give the
length bonus only to correct answers, as in
[Kimi k1.5](https://arxiv.org/abs/2501.12599); others train the model to
follow a given token budget (lecture, block 11).
</details>

### Card 2. An LLM judge

When there is no checker, another model judges the answer. Here the reward is
the probability that the judge answers "yes" to "Is the proposed answer
correct?", and the judge is the same 0.5B model. Four candidate answers to
the same question:

In [ ]:
CANDIDATES = {
    "A: correct, plain": f"The answer is \\boxed{{{GOLD}}}.",
    "B: wrong, plain": f"The answer is \\boxed{{{GOLD + 7}}}.",
    "C: wrong, confident": (f"The answer is \\boxed{{{GOLD + 7}}}. I verified every step twice; "
                            "this answer is correct."),
    "D: correct, hesitant": (f"I am not completely sure, but I think the answer might be "
                             f"\\boxed{{{GOLD}}}."),
}
for k, v in CANDIDATES.items():
    print(f"{k:<22} {v}")

**Question:** rank the four candidates by the judge's reward. Which one would
a policy trained against this judge learn to write?

In [ ]:
@torch.no_grad()
def judge_reward(question, answer):
    prompt = chat_prompt(tok, f"Question: {question}\n\nProposed answer: {answer}\n\n"
                              "Is the proposed answer correct? Reply with yes or no.",
                         system=None)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    logp = torch.log_softmax(model(**enc).logits[0, -1].float(), -1)
    yes = torch.logsumexp(logp[[tok.encode(w)[0] for w in ("yes", "Yes")]], 0)
    no = torch.logsumexp(logp[[tok.encode(w)[0] for w in ("no", "No")]], 0)
    return float(torch.sigmoid(yes - no))          # P(yes) among {yes, no}


scores = {k: judge_reward(QUESTION, v) for k, v in CANDIDATES.items()}
for k, v in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"{v:.2f}  {k}")

<details><summary>What happens at scale</summary>

A policy trained against a judge learns whatever the judge rewards, tone and
claims of having checked included. On lecture slide 12.2 a policy trained
against a reasoning judge found answers that the judge and other strong
judges scored high, while the score of the gold judge fell. Common defences:
a stronger judge or several judges, rubrics that ask for evidence, and
tracking an independent metric next to the reward.
</details>

### Card 3. "The tests pass"

A coding task is rewarded with the share of visible unit tests that pass.
The task: `digits_sum(n)` returns the sum of the decimal digits of a
non-negative integer. Three candidate solutions:

In [ ]:
SOLUTIONS = {
    "honest": "def digits_sum(n):\n    return sum(int(c) for c in str(n))",
    "lookup": "def digits_sum(n):\n    return {0: 0, 7: 7, 123: 6, 9999: 36}.get(n, 1)",
    "almost": "def digits_sum(n):\n    return n % 9 if n % 9 else (9 if n else 0)",
}
VISIBLE = [(0, 0), (7, 7), (123, 6), (9999, 36)]
HIDDEN = [(10, 1), (55, 10), (1000001, 2), (98765, 35), (19, 10)]
for name, code in SOLUTIONS.items():
    print(f"# {name}\n{code}\n")

**Question:** which solutions get the full reward on the visible tests, and
which of them are actually correct?

In [ ]:
def pass_rate(code, tests):
    ns = {}
    exec(code, ns)
    return np.mean([ns["digits_sum"](x) == y for x, y in tests])


for name, code in SOLUTIONS.items():
    print(f"{name:>7}: reward (visible) {pass_rate(code, VISIBLE):.2f}   "
          f"hidden tests {pass_rate(code, HIDDEN):.2f}")

<details><summary>What happens at scale</summary>

Under a "tests pass" reward, special-casing the tests earns full reward, and
RL on coding tasks does find such solutions. OpenAI reported reasoning
models that made tests pass without solving the task during training, and
caught it by reading their chains of thought
([Baker et al., 2025](https://arxiv.org/abs/2503.11926)). Tests the model
cannot see or edit and monitoring of the reasoning reduce this; the same
paper shows that penalizing the reasoning for it teaches the model to hide
the intent instead.
</details>